# Demo Thực Nghiệm Chương 3: Ứng Dụng Thuật Toán CHARM (Group_12_Charm)

**Môn học:** Khai phá Dữ liệu (Data Mining) — Khoa CNTT, Trường ĐH Khoa học Tự nhiên, ĐHQG-HCM (FIT-HCMUS)  
**Nhóm thực hiện:** Nhóm 12 (`Group_12_Charm`)  
**Repository:** [`khoagiap1312/Group_12_Charm`](https://github.com/khoagiap1312/Group_12_Charm)  
**Kho dữ liệu Hugging Face:** [`Quanbro7/Charm_Dataset`](https://huggingface.co/datasets/Quanbro7/Charm_Dataset)

---

### Mục tiêu & Luồng thực thi của Notebook
Notebook này gọi **trực tiếp các kịch bản chính thức trong repo của nhóm** (không dùng thư viện FIM ngoài, không dùng fallback ẩn; mọi sự cố sẽ in rõ lỗi `stdout`/`stderr` và dừng ngay):

1. **Phần 1 — Thiết lập môi trường:** Cài đặt **Julia 1.13.4** (khớp `Manifest.toml`), clone repo `Group_12_Charm`, cài thư viện Python cần thiết và khởi tạo `Project.toml`.
2. **Phần 2 — Tải dữ liệu thực tế:** Tải trực tiếp dữ liệu tiền xử lý của 2 ứng dụng từ `Quanbro7/Charm_Dataset` vào `data/application/...`.
3. **Phần 3 — Ứng dụng 1 (Web Server Access Logs):** Gọi `src/main.jl` (CHARM tối ưu của nhóm), `scripts/invert_parse.jl` (giải mã nhãn sự kiện) và `scripts/analyze_web_logs.py` (thống kê, so sánh Normal vs. Anomalous và vẽ biểu đồ).
4. **Phần 4 — Ứng dụng 2 (Caltech-101 Visual Patterns):** Gọi `scripts/analyze_caltech101.py --split-classes`, chạy `src/main.jl` cho toàn tập và 3 lớp riêng, sau đó gọi `scripts/analyze_caltech101.py` để tính Class Purity và vẽ lưới patch 3×3.
5. **Phần 5 — Kiểm tra tổng hợp đầu ra:** Xác minh toàn bộ tệp CSV và hình PNG đã được tạo đầy đủ.

## Phần 1. Thiết lập môi trường trên Google Colab (Julia + Python + Mã nguồn)

### 1.1. Cấu hình tham số tái lập và hàm chạy lệnh có in chi tiết lỗi
Hàm `run_cmd` được dùng xuyên suốt notebook: nếu bất kỳ script Julia/Python nào gặp sự cố, notebook sẽ in rõ câu lệnh, mã lỗi (`returncode`), `STDOUT` và `STDERR` rồi dừng ngay để dễ kiểm tra.

In [ ]:
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

# ==============================================================================
# CẤU HÌNH CHUNG (CỐ ĐỊNH SEED = 0)
# ==============================================================================
SEED = 0
random.seed(SEED)
np.random.seed(SEED)

GITHUB_REPO_URL = "https://github.com/khoagiap1312/Group_12_Charm.git"
REPO_DIR_NAME = "Group_12_Charm"
HF_DATASET_REPO = "Quanbro7/Charm_Dataset"

# Khớp đúng julia_version = "1.13.4" trong Manifest.toml của nhóm
JULIA_VERSION = "1.13.4"
JULIA_MAJOR_MINOR = "1.13"


def run_cmd(cmd: list[str], desc: str = "") -> subprocess.CompletedProcess:
    '''Chạy lệnh hệ thống; in đầy đủ STDOUT/STDERR và dừng ngay nếu xảy ra lỗi.'''
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    dt = time.perf_counter() - t0
    if proc.returncode != 0:
        print(f"[ERROR] Thất bại khi thực thi ({desc or 'command'}) — Exit code: {proc.returncode}")
        print(f"  Command: {' '.join(cmd)}")
        if proc.stdout.strip():
            print("  --- STDOUT ---\n" + proc.stdout.strip())
        if proc.stderr.strip():
            print("  --- STDERR ---\n" + proc.stderr.strip())
        raise RuntimeError(f"Lệnh thất bại (exit code {proc.returncode}): {' '.join(cmd)}")
    if desc:
        print(f"[OK] {desc} ({dt:.2f}s)")
    return proc


print(f"Python version : {sys.version.split()[0]}")
print(f"Fixed SEED     : {SEED}")
print(f"GitHub Repo    : {GITHUB_REPO_URL}")
print(f"HF Dataset Repo: {HF_DATASET_REPO}")

### 1.2. Tự động tải và cài đặt Julia 1.13.4 trên Google Colab
Ô lệnh kiểm tra xem `julia` đã có sẵn chưa; nếu chưa, tự tải bản nhị phân chính thức **Julia 1.13.4 (Linux x86_64)** để khớp hoàn toàn với `Manifest.toml` của nhóm.

In [ ]:
%%bash -s "$JULIA_VERSION" "$JULIA_MAJOR_MINOR"
set -euo pipefail
JULIA_VER=$1
JULIA_MM=$2

if command -v julia >/dev/null 2>&1; then
    echo "[OK] Julia đã được cài đặt sẵn:"
    julia --version
else
    echo "[INFO] Đang tải Julia ${JULIA_VER} cho Linux x86_64..."
    URL="https://julialang-s3.julialang.org/bin/linux/x64/${JULIA_MM}/julia-${JULIA_VER}-linux-x86_64.tar.gz"
    wget -q --show-progress -O /tmp/julia.tar.gz "${URL}"
    mkdir -p /usr/local/julia
    tar -xzf /tmp/julia.tar.gz -C /usr/local/julia --strip-components=1
    ln -sf /usr/local/julia/bin/julia /usr/local/bin/julia
    rm -f /tmp/julia.tar.gz
    echo "[OK] Cài đặt hoàn tất:"
    julia --version
fi

### 1.3. Clone mã nguồn `Group_12_Charm` và cài đặt môi trường
- Cài các thư viện Python dùng trong `scripts/analyze_web_logs.py`, `scripts/analyze_caltech101.py` và tải dữ liệu Hugging Face.
- Clone repo `Group_12_Charm` (hoặc dùng thư mục hiện tại nếu đang chạy trong repo).
- Khởi tạo gói Julia theo `Project.toml` / `Manifest.toml` (đặt `JULIA_CONDAPKG_BACKEND=Null` để không kéo thêm Conda phụ trên Colab) và kiểm tra CLI `src/main.jl`.

In [ ]:
# 1. Cài đặt thư viện Python cần thiết cho các script trong repo
run_cmd(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "huggingface_hub", "pandas", "numpy", "matplotlib", "pillow",
    ],
    desc="Cài đặt thư viện Python",
)

# 2. Xác định hoặc clone thư mục gốc Group_12_Charm
cwd = Path.cwd()
if (cwd / "src" / "main.jl").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src" / "main.jl").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd / REPO_DIR_NAME / "src" / "main.jl").exists():
    PROJECT_ROOT = cwd / REPO_DIR_NAME
else:
    run_cmd(["git", "clone", "--depth", "1", GITHUB_REPO_URL, REPO_DIR_NAME], desc="Clone repo Group_12_Charm")
    PROJECT_ROOT = cwd / REPO_DIR_NAME

os.chdir(PROJECT_ROOT)
print(f"[OK] Thư mục làm việc hiện tại: {Path.cwd()}")

# 3. Khởi tạo môi trường Julia theo Project.toml / Manifest.toml
os.environ["JULIA_CONDAPKG_BACKEND"] = "Null"
assert (PROJECT_ROOT / "Project.toml").exists(), "Không tìm thấy Project.toml trong thư mục gốc repo!"
run_cmd(
    ["julia", "--project=.", "-e", "using Pkg; Pkg.instantiate()"],
    desc="Khởi tạo môi trường Julia (Pkg.instantiate)",
)

# 4. Kiểm tra các script chính thức cần gọi trong repo
required_scripts = [
    "src/main.jl",
    "scripts/invert_parse.jl",
    "scripts/analyze_web_logs.py",
    "scripts/analyze_caltech101.py",
]
for s in required_scripts:
    assert Path(s).exists(), f"Thiếu script bắt buộc trong repo: {s}"

res_help = run_cmd(["julia", "--project=.", "src/main.jl", "--help"], desc="Kiểm tra CLI src/main.jl")
print(res_help.stdout)

## Phần 2. Tải dữ liệu thực tế từ Hugging Face (`Quanbro7/Charm_Dataset`)

Tải trực tiếp các tệp đã tiền xử lý của hai ứng dụng về đúng thư mục mà `scripts/analyze_web_logs.py` và `scripts/analyze_caltech101.py` yêu cầu:
- `data/application/web_server_access_logs/pre_process/` (`transaction.txt`, `mapping.csv`)
- `data/application/cal_tech_101/pre_process/` (`transactions.txt`, `image_mapping.csv`, `visual_word_stats.csv`, `patches/word_*.png`)

Nếu thiếu bất kỳ tệp nào trên Hugging Face, notebook sẽ báo lỗi rõ ràng và dừng ngay (không dùng dữ liệu giả hay fallback).

In [ ]:
import shutil
import pandas as pd
from huggingface_hub import HfApi, snapshot_download

WEB_PRE_DIR = Path("data/application/web_server_access_logs/pre_process")
CALTECH_PRE_DIR = Path("data/application/cal_tech_101/pre_process")
WEB_PRE_DIR.mkdir(parents=True, exist_ok=True)
CALTECH_PRE_DIR.mkdir(parents=True, exist_ok=True)

api = HfApi()
hf_files = api.list_repo_files(repo_id=HF_DATASET_REPO, repo_type="dataset")
print(f"[OK] Kết nối Hugging Face dataset '{HF_DATASET_REPO}' ({len(hf_files)} tệp).")


def download_hf_dataset_dir(keyword: str, local_dir: Path, required_files: list[str]) -> None:
    '''Tải đúng thư mục dữ liệu tiền xử lý từ Hugging Face về local_dir; báo lỗi ngay nếu thiếu.'''
    if all((local_dir / f).exists() for f in required_files):
        print(f"[OK] Dữ liệu đã có sẵn tại {local_dir}")
        return

    anchor_hits = [
        f for f in hf_files
        if keyword in f.lower() and Path(f).name == required_files[0]
    ]
    if not anchor_hits:
        raise FileNotFoundError(
            f"Không tìm thấy '{required_files[0]}' (từ khóa '{keyword}') trên Hugging Face repo '{HF_DATASET_REPO}'."
        )

    prefix = str(Path(anchor_hits[0]).parent)
    allow_patterns = [f"{prefix}/*"] if prefix != "." else required_files
    print(f"[INFO] Đang tải '{prefix}' từ {HF_DATASET_REPO} -> {local_dir} ...")

    snap_dir = Path(
        snapshot_download(
            repo_id=HF_DATASET_REPO,
            repo_type="dataset",
            allow_patterns=allow_patterns,
        )
    )
    src_root = snap_dir / prefix if prefix != "." else snap_dir
    for item in src_root.rglob("*"):
        if item.is_file():
            dst = local_dir / item.relative_to(src_root)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, dst)

    missing = [f for f in required_files if not (local_dir / f).exists()]
    if missing:
        raise FileNotFoundError(f"Thiếu tệp bắt buộc sau khi tải về {local_dir}: {missing}")
    print(f"[OK] Đã tải đủ dữ liệu về {local_dir}")


# 1. Tải dữ liệu Ứng dụng 1: Web Server Access Logs
download_hf_dataset_dir(
    keyword="web",
    local_dir=WEB_PRE_DIR,
    required_files=["transaction.txt", "mapping.csv"],
)

# 2. Tải dữ liệu Ứng dụng 2: Caltech-101
download_hf_dataset_dir(
    keyword="cal",
    local_dir=CALTECH_PRE_DIR,
    required_files=["transactions.txt", "image_mapping.csv", "visual_word_stats.csv"],
)

patch_files = list((CALTECH_PRE_DIR / "patches").glob("word_*.png"))
if not patch_files:
    raise FileNotFoundError(
        f"Không tìm thấy ảnh patch trong {CALTECH_PRE_DIR / 'patches'}. "
        "Kiểm tra lại thư mục patches/ trên Hugging Face."
    )
print(f"[OK] Đã tìm thấy {len(patch_files)} ảnh lưới patch đại diện tại {CALTECH_PRE_DIR / 'patches'}")

---
## Phần 3. Ứng dụng 1: Khai phá mẫu hành vi và phát hiện bất thường trên Web Server Access Logs

### Quy trình gọi trực tiếp các script trong repo
1. `julia --project=. src/main.jl -a optimized`: Khai phá Closed Frequent Itemsets trên **520.428 phiên truy cập** ở 5 mức `minsup ∈ {0.1, 0.05, 0.02, 0.01, 0.005}`.
2. `julia --project=. scripts/invert_parse.jl`: Giải mã kết quả SPMF sang tên nhãn sự kiện (`mapping.csv`) tại `minsup = 0.01` và `0.005`.
3. `python scripts/analyze_web_logs.py`: Tổng hợp `minsup_summary.csv`, đối chiếu Normal vs. Anomalous (`normal_vs_anomaly_comparison.csv`) và vẽ 2 biểu đồ.

In [ ]:
WEB_OUT_DIR = Path("results/application/web_server_access_logs")
WEB_OUT_DIR.mkdir(parents=True, exist_ok=True)

WEB_MINSUPS = ["0.1", "0.05", "0.02", "0.01", "0.005"]

print("=== Bước 3.1: Chạy CHARM tối ưu (src/main.jl) trên Web Server Access Logs ===")
for ms in WEB_MINSUPS:
    out_file = WEB_OUT_DIR / f"charm_minsup_{ms}.txt"
    res = run_cmd(
        [
            "julia", "--project=.", "src/main.jl",
            "-i", str(WEB_PRE_DIR / "transaction.txt"),
            "-m", ms,
            "-o", str(out_file),
            "-a", "optimized",
        ],
        desc=f"CHARM Web Logs minsup={ms}",
    )
    print(" ", res.stdout.strip())

In [ ]:
print("=== Bước 3.2: Giải mã SPMF sang Event Templates bằng scripts/invert_parse.jl ===")
for ms in ["0.01", "0.005"]:
    spmf_in = WEB_OUT_DIR / f"charm_minsup_{ms}.txt"
    decoded_out = WEB_OUT_DIR / f"decoded_minsup_{ms}.txt"
    res = run_cmd(
        [
            "julia", "--project=.", "scripts/invert_parse.jl",
            "--spmf", str(spmf_in),
            "--map", str(WEB_PRE_DIR / "mapping.csv"),
            "--output", str(decoded_out),
        ],
        desc=f"invert_parse.jl minsup={ms}",
    )
    print(" ", res.stdout.strip())

print("\nMẫu 5 dòng đầu trong decoded_minsup_0.005.txt:")
with open(WEB_OUT_DIR / "decoded_minsup_0.005.txt", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(" ", f.readline().strip())

In [ ]:
from IPython.display import Image, display

print("=== Bước 3.3: Phân tích & vẽ biểu đồ bằng scripts/analyze_web_logs.py ===")
res = run_cmd([sys.executable, "scripts/analyze_web_logs.py"], desc="analyze_web_logs.py")
print(res.stdout)

print("\n=== Bảng tổng hợp số lượng Closed Itemsets theo minsup ===")
display(pd.read_csv(WEB_OUT_DIR / "minsup_summary.csv"))

df_web_cmp = pd.read_csv(WEB_OUT_DIR / "normal_vs_anomaly_comparison.csv")
print("\n=== Top 10 Closed Itemsets đặc trưng mạnh cho phiên Bất thường / Bot / Scan (Ratio >= 1.5) ===")
df_anom_top = (
    df_web_cmp[df_web_cmp["anom_to_norm_ratio"] >= 1.5]
    .sort_values(["sup_anomaly_pct", "anom_to_norm_ratio"], ascending=[False, False])
    [["length", "sup_all_pct", "sup_normal_pct", "sup_anomaly_pct", "anom_to_norm_ratio", "event_templates"]]
    .head(10)
)
display(df_anom_top)

for fig_name in ["fig1_minsup_vs_itemsets.png", "fig2_normal_vs_anomaly.png"]:
    fig_path = WEB_OUT_DIR / "figures" / fig_name
    assert fig_path.exists(), f"Không tìm thấy hình kết quả: {fig_path}"
    print(f"\n--- {fig_name} ---")
    display(Image(filename=str(fig_path), width=820))

### Diễn giải nhanh kết quả Ứng dụng 1 (Web Server Access Logs)
- **Độ nhạy ngưỡng `minsup`:** Khi hạ `minsup` từ `10%` xuống `0,5%`, tổng số tập mục đóng tăng phi tuyến **43,8 lần** (từ **179** lên **7.839** tập đóng) và độ dài cực đại tăng gấp đôi (từ **6** lên **12** sự kiện đồng xuất hiện).
- **Tách biệt rõ rệt hành vi người dùng thật và Bot/Scraper:**
  - *Phiên bình thường (Normal):* Chiếm ưu thế ở các tổ hợp tải đầy đủ tài nguyên đi kèm điều hướng và trình duyệt di động (`GET_IMAGE_200 + UA_MOBILE`, `REF_LINKED + UA_MOBILE`).
  - *Phiên bất thường (Anomalous):* Nổi bật với các chữ ký bot/scraper gần như chỉ xuất hiện ở nhóm bất thường (`Normal = 0%`) như `REF_DIRECT + UA_KNOWN_BOT`, `BEHAVIOR_NO_ASSET_SCRAPING + REF_DIRECT` và `BEHAVIOR_NO_ASSET_SCRAPING + REF_DIRECT + UA_KNOWN_BOT`.
- **Hạn chế:** Tập log thô không có nhãn ground-truth nên việc gán nhãn dựa trên heuristic; đồng thời mô hình tập mục đóng bỏ qua thứ tự thời gian giữa các request trong cùng cửa sổ 5 phút.

---
## Phần 4. Ứng dụng 2: Khai phá mẫu trực quan (Visual Words) trên tập ảnh Caltech-101

### Quy trình gọi trực tiếp các script trong repo
1. `python scripts/analyze_caltech101.py --split-classes`: Tách `transactions.txt` (600 ảnh, 300 visual words từ RootSIFT + K-Means `SEED = 0`) thành 3 tệp giao dịch theo từng lớp (`airplanes`, `faces_easy`, `motorbikes`).
2. `julia --project=. src/main.jl -a optimized`: Khai phá Closed Itemsets cho toàn tập `all` (`minsup ∈ {0.10, 0.07, 0.05, 0.03}`) và từng lớp (`minsup ∈ {0.15, 0.10, 0.08}`).
3. `python scripts/analyze_caltech101.py`: Tính độ tinh khiết theo lớp (**Class Purity**), xuất các bảng CSV và vẽ biểu đồ `minsup` + 4 lưới patch `3×3` đại diện.

In [ ]:
CALTECH_OUT_DIR = Path("results/application/cal_tech_101")
CALTECH_OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Tách giao dịch theo từng lớp bằng scripts/analyze_caltech101.py --split-classes
print("=== Bước 4.1: Tách file giao dịch theo từng lớp ===")
res_split = run_cmd(
    [sys.executable, "scripts/analyze_caltech101.py", "--split-classes"],
    desc="analyze_caltech101.py --split-classes",
)
print(res_split.stdout)

# 2. Chạy CHARM tối ưu (src/main.jl) trên phạm vi toàn bộ (all) và 3 lớp riêng
RUN_PLAN = [
    ("all", CALTECH_PRE_DIR / "transactions.txt", ["0.10", "0.07", "0.05", "0.03"]),
    ("airplanes", CALTECH_PRE_DIR / "transactions_airplanes.txt", ["0.15", "0.10", "0.08"]),
    ("faces_easy", CALTECH_PRE_DIR / "transactions_faces_easy.txt", ["0.15", "0.10", "0.08"]),
    ("motorbikes", CALTECH_PRE_DIR / "transactions_motorbikes.txt", ["0.15", "0.10", "0.08"]),
]

print("=== Bước 4.2: Chạy CHARM tối ưu (src/main.jl) trên Caltech-101 ===")
for scope, tx_file, minsups in RUN_PLAN:
    for ms in minsups:
        out_file = CALTECH_OUT_DIR / f"charm_{scope}_minsup_{ms}.txt"
        res = run_cmd(
            [
                "julia", "--project=.", "src/main.jl",
                "-i", str(tx_file),
                "-m", ms,
                "-o", str(out_file),
                "-a", "optimized",
            ],
            desc=f"CHARM Caltech {scope} minsup={ms}",
        )
        print(" ", res.stdout.strip())

In [ ]:
print("=== Bước 4.3: Phân tích Class Purity & vẽ lưới patch bằng scripts/analyze_caltech101.py ===")
res = run_cmd([sys.executable, "scripts/analyze_caltech101.py"], desc="analyze_caltech101.py")
print(res.stdout)

print("\n=== Bảng tổng hợp Closed Itemsets theo phạm vi và minsup ===")
display(pd.read_csv(CALTECH_OUT_DIR / "caltech_minsup_summary.csv"))

for scope in ["all", "airplanes", "faces_easy", "motorbikes"]:
    csv_path = CALTECH_OUT_DIR / f"top_visual_itemsets_{scope}.csv"
    assert csv_path.exists(), f"Không tìm thấy bảng kết quả: {csv_path}"
    print(f"\n=== Top 8 Closed Itemsets (len >= 2) — Phạm vi: {scope.upper()} ===")
    display(pd.read_csv(csv_path).head(8))

In [ ]:
caltech_figs = [
    ("fig1_caltech_minsup_curve.png", "Biểu đồ ảnh hưởng của minsup lên số lượng Closed Itemsets (len >= 2)"),
    ("fig_patches_all.png", "Lưới patch 3×3 — Top 5 Closed Itemsets trên toàn bộ 600 ảnh (all, minsup=0.03)"),
    ("fig_patches_airplanes.png", "Lưới patch 3×3 — Top 5 Closed Itemsets lớp AIRPLANES (minsup=0.08)"),
    ("fig_patches_faces_easy.png", "Lưới patch 3×3 — Top 5 Closed Itemsets lớp FACES_EASY (minsup=0.08)"),
    ("fig_patches_motorbikes.png", "Lưới patch 3×3 — Top 5 Closed Itemsets lớp MOTORBIKES (minsup=0.08)"),
]

for fname, caption in caltech_figs:
    fpath = CALTECH_OUT_DIR / "figures" / fname
    assert fpath.exists(), f"Không tìm thấy hình kết quả: {fpath}"
    print(f"\n--- {caption} ({fname}) ---")
    display(Image(filename=str(fpath), width=850))

### Diễn giải nhanh kết quả Ứng dụng 2 (Caltech-101 Visual Patterns)
- **Bùng nổ tổ hợp & khác biệt nội lớp:** Trên toàn tập `all`, hạ `minsup` từ `10%` xuống `3%` khiến số closed itemset (`len ≥ 2`) tăng từ **3** lên **15.545** (gấp >5.181 lần) do độ dày ma trận đạt `15,97%`. Ở `minsup = 8%` theo lớp, `motorbikes` sinh nhiều tập đóng nhất (**2.405**), tiếp đến `faces_easy` (**1.566**) và `airplanes` (**865**).
- **Quy luật Độ dài – Độ tinh khiết (Length–Purity Trade-off):** Các tổ hợp độ dài 4–5 như `{26, 45, 56, 102, 259}` (`motorbikes`), `{46, 97, 141, 167, 242}` (`faces_easy`) và `{5, 14, 200, 239}` (`airplanes`) đạt độ tinh khiết tuyệt đối **100,0%**, đóng vai trò đặc trưng bậc trung (*visual phrases*) hữu ích cho **Phân loại ảnh** và **Truy vấn ảnh (CBIR)**.
- **Ảnh hưởng tham số `k` và hạn chế:** `k = 300` giữ cân bằng giữa lượng tử hóa thô (`k` nhỏ) và phân mảnh (`k` lớn). Hạn chế chính là bỏ qua vị trí không gian tương đối giữa các patch và nhạy cảm với nhiễu nền chung (như `{157, 191}`, purity **47,3%**).

---
## Phần 5. Kiểm tra tổng hợp đầu ra (Reproducibility Checklist)

Xác nhận toàn bộ 14 tệp kết quả (`.csv` và `.png`) của cả hai ứng dụng đã được tạo đầy đủ và khác rỗng.

In [ ]:
expected_outputs = [
    WEB_OUT_DIR / "minsup_summary.csv",
    WEB_OUT_DIR / "normal_vs_anomaly_comparison.csv",
    WEB_OUT_DIR / "figures" / "fig1_minsup_vs_itemsets.png",
    WEB_OUT_DIR / "figures" / "fig2_normal_vs_anomaly.png",
    CALTECH_OUT_DIR / "caltech_minsup_summary.csv",
    CALTECH_OUT_DIR / "top_visual_itemsets_all.csv",
    CALTECH_OUT_DIR / "top_visual_itemsets_airplanes.csv",
    CALTECH_OUT_DIR / "top_visual_itemsets_faces_easy.csv",
    CALTECH_OUT_DIR / "top_visual_itemsets_motorbikes.csv",
    CALTECH_OUT_DIR / "figures" / "fig1_caltech_minsup_curve.png",
    CALTECH_OUT_DIR / "figures" / "fig_patches_all.png",
    CALTECH_OUT_DIR / "figures" / "fig_patches_airplanes.png",
    CALTECH_OUT_DIR / "figures" / "fig_patches_faces_easy.png",
    CALTECH_OUT_DIR / "figures" / "fig_patches_motorbikes.png",
]

rows = []
missing = []
for p in expected_outputs:
    exists = p.exists() and p.stat().st_size > 0
    if not exists:
        missing.append(str(p))
    rows.append({
        "Tệp kết quả": str(p),
        "Trạng thái": "OK" if exists else "THIẾU",
        "Kích thước (KB)": round(p.stat().st_size / 1024.0, 1) if exists else 0.0,
    })

display(pd.DataFrame(rows))
if missing:
    raise FileNotFoundError(f"Thiếu {len(missing)} tệp kết quả: {missing}")
print("\n[SUCCESS] Toàn bộ pipeline Demo Chương 3 (2 ứng dụng) đã chạy sạch và tái lập đầy đủ!")